# Day 2: Advanced Chunking & ChromaDB

In this session, we will:
1. Experiment with different chunking strategies.
2. Use ChromaDB as a vector database.
3. Ingest a larger dataset (PDF or text files).
4. Build a more robust RAG pipeline.

## Prerequisites
```bash
pip install litellm chromadb pypdf
```

In [1]:
!pip install litellm chromadb pypdf

  Using cached posthog-5.4.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached opentelemetry_api-1.39.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.39.1-py3-none-any.whl.metadata (2.5 kB)
  Using cached opentelemetry_sdk-1.39.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached importlib_resources-6.5.2-py3-none-any.whl.metadata (3.9 kB)
  Using cached grpcio-1.78.0-cp311-cp311-macosx_11_0_universal2.whl.metadata (3.8 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-macosx_10_12_universal2.whl.metadata (10 kB)
  Using cached mmh3-5.2.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (14 kB)
  Using cached backoff-2.2.1-py3-none-any.whl.metadata (14 kB)
  Using cached pyproject_hooks-1.2.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached du

In [1]:
import chromadb
from chromadb.config import Settings
from litellm import completion, embedding
import os

## 1. Advanced Chunking Strategies

Let's implement a recursive character text splitter.

In [2]:
class RecursiveCharacterTextSplitter:
    def __init__(self, chunk_size=500, chunk_overlap=50, separators=None):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.separators = separators or ["\n\n", "\n", ". ", " ", ""]
    
    def split_text(self, text):
        chunks = []
        
        def _split(text, separators):
            if not separators:
                return [text]
            
            separator = separators[0]
            splits = text.split(separator) if separator else list(text)
            
            current_chunk = ""
            result = []
            
            for split in splits:
                if len(current_chunk) + len(split) <= self.chunk_size:
                    current_chunk += split + separator
                else:
                    if current_chunk:
                        result.append(current_chunk.strip())
                    
                    if len(split) > self.chunk_size:
                        # Split further with next separator
                        result.extend(_split(split, separators[1:]))
                        current_chunk = ""
                    else:
                        current_chunk = split + separator
            
            if current_chunk:
                result.append(current_chunk.strip())
            
            return result
        
        return _split(text, self.separators)

# Test the splitter
sample_text = """
Artificial Intelligence has revolutionized many industries. Machine learning, a subset of AI, enables computers to learn from data.

Deep learning, which uses neural networks, has achieved remarkable results in image recognition and natural language processing.

Large Language Models like GPT-4 can generate human-like text. However, they have limitations such as knowledge cutoffs and hallucinations.

Retrieval-Augmented Generation (RAG) addresses these limitations by combining LLMs with external knowledge bases.
"""

splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)
chunks = splitter.split_text(sample_text)

print(f"Created {len(chunks)} chunks:\n")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1} ({len(chunk)} chars): {chunk}\n")

Created 4 chunks:

Chunk 1 (131 chars): Artificial Intelligence has revolutionized many industries. Machine learning, a subset of AI, enables computers to learn from data.

Chunk 2 (128 chars): Deep learning, which uses neural networks, has achieved remarkable results in image recognition and natural language processing.

Chunk 3 (139 chars): Large Language Models like GPT-4 can generate human-like text. However, they have limitations such as knowledge cutoffs and hallucinations.

Chunk 4 (113 chars): Retrieval-Augmented Generation (RAG) addresses these limitations by combining LLMs with external knowledge bases.



## 2. Setting up ChromaDB

ChromaDB is an embedded vector database that's perfect for prototyping.

In [3]:
# Initialize ChromaDB client
client = chromadb.Client(Settings(
    persist_directory="/Users/rajesh/Desktop/rajesh/Archive/teaching/RAG_sessions/Day-2/chroma_db"
))


# Create or get a collection
collection_name = "rag_workshop_day2"


# Delete collection if it exists (for clean start)
try:
    client.delete_collection(name=collection_name)
except:
    pass

collection = client.create_collection(
    name=collection_name,
    metadata={"description": "Day 2 RAG Workshop Collection"}
)

print(f"Collection '{collection_name}' created successfully!")

Collection 'rag_workshop_day2' created successfully!


## 3. Embedding Function with LiteLM

We'll create a custom embedding function for ChromaDB.

In [4]:
def get_embeddings(texts, model="ollama/nomic-embed-text"):
    """Get embeddings for a list of texts using LiteLM"""
    try:
        response = embedding(model=model, input=texts)
        return [item['embedding'] for item in response['data']]
    except Exception as e:
        print(f"Error getting embeddings: {e}")
        return []

# Test
test_embeddings = get_embeddings(["Hello world", "Test embedding"])
print(f"Generated {len(test_embeddings)} embeddings, each with {len(test_embeddings[0])} dimensions")

Generated 2 embeddings, each with 768 dimensions


## 4. Ingesting Documents into ChromaDB

Let's create a larger knowledge base about AI and RAG.

In [6]:
# Sample documents (in practice, you'd load these from files)
documents = [
    """Retrieval-Augmented Generation (RAG) is a technique that enhances large language models by retrieving relevant information from external knowledge bases. This approach helps reduce hallucinations and provides more accurate, up-to-date responses.""",
    
    """Vector databases store high-dimensional embeddings and enable fast similarity search. Popular options include ChromaDB, Pinecone, Weaviate, and FAISS. These databases use algorithms like HNSW for efficient nearest neighbor search.""",
    
    """Chunking strategies significantly impact RAG performance. Fixed-size chunking is simple but may break semantic boundaries. Recursive chunking preserves document structure. Semantic chunking groups related content but is computationally expensive.""",
    
    """Embeddings are dense vector representations of text that capture semantic meaning. Models like OpenAI's text-embedding-ada-002, Cohere's embed-v3, and open-source options like nomic-embed-text convert text into numerical vectors.""",
    
    """Hybrid search combines dense vector search with sparse keyword search (BM25). This approach leverages the strengths of both methods: semantic understanding from embeddings and exact matching from keywords.""",
    
    """LiteLM is a unified interface for calling 100+ LLMs using the OpenAI format. It supports providers like OpenAI, Anthropic, Cohere, Ollama, and more. This makes it easy to switch between different models without changing code.""",
]



# Chunk the documents
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
all_chunks = []
metadata_list = []

for doc_id, doc in enumerate(documents):
    doc_chunks = splitter.split_text(doc)
    for chunk_id, chunk in enumerate(doc_chunks):
        all_chunks.append(chunk)
        metadata_list.append({
            "doc_id": doc_id,
            "chunk_id": chunk_id,
            "source": f"document_{doc_id}"
        })

print(f"Total chunks: {len(all_chunks)}")

# Generate embeddings
print("Generating embeddings...")
embeddings = get_embeddings(all_chunks)

# Add to ChromaDB
collection.add(
    embeddings=embeddings,
    documents=all_chunks,
    metadatas=metadata_list,
    ids=[f"chunk_{i}" for i in range(len(all_chunks))]
)

print(f"Successfully indexed {len(all_chunks)} chunks into ChromaDB!")




Total chunks: 12
Generating embeddings...
Successfully indexed 12 chunks into ChromaDB!


## 5. Querying ChromaDB

Now let's retrieve relevant chunks for a query.

In [7]:
def query_chromadb(query_text, n_results=3):
    """Query ChromaDB for relevant chunks"""
    # Get query embedding
    query_embedding = get_embeddings([query_text])[0]
    
    # Query the collection
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )
    
    return results

# Test query
query = "What are the benefits of hybrid search?"
results = query_chromadb(query, n_results=3)

print(f"Query: {query}\n")
print("Retrieved chunks:")
for i, (doc, metadata, distance) in enumerate(zip(
    results['documents'][0],
    results['metadatas'][0],
    results['distances'][0]
)):
    print(f"\n{i+1}. (Distance: {distance:.4f})")
    print(f"   Source: {metadata['source']}")
    print(f"   Text: {doc}")

Query: What are the benefits of hybrid search?

Retrieved chunks:

1. (Distance: 0.7063)
   Source: document_4
   Text: Hybrid search combines dense vector search with sparse keyword search (BM25).

2. (Distance: 0.7495)
   Source: document_1
   Text: Vector databases store high-dimensional embeddings and enable fast similarity search. Popular options include ChromaDB, Pinecone, Weaviate, and FAISS.

3. (Distance: 0.7933)
   Source: document_1
   Text: These databases use algorithms like HNSW for efficient nearest neighbor search..


## 6. Complete RAG Pipeline with ChromaDB

In [8]:
def rag_with_chromadb(question, n_results=3, llm_model="ollama/llama3.2:1b"):
    """Complete RAG pipeline using ChromaDB"""
    print(f"Question: {question}\n")
    
    # 1. Retrieve relevant chunks
    results = query_chromadb(question, n_results=n_results)
    context_chunks = results['documents'][0]
    
    print("Retrieved Context:")
    for i, chunk in enumerate(context_chunks):
        print(f"{i+1}. {chunk[:100]}...")
    
    # 2. Build prompt
    context = "\n\n".join(context_chunks)
    prompt = f"""Answer the question based on the provided context. If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {question}

Answer:"""
    
    # 3. Generate answer
    response = completion(
        model=llm_model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    answer = response['choices'][0]['message']['content']
    print(f"\nAnswer:\n{answer}")
    return answer

# Test the complete pipeline
rag_with_chromadb("What is LiteLM and what does it support?")
print("\n" + "="*80 + "\n")
rag_with_chromadb("Explain different chunking strategies")
print("\n" + "="*80 + "\n")
rag_with_chromadb("What is the capital of France?")  # Should say "I don't know"

Question: What is LiteLM and what does it support?

Retrieved Context:
1. LiteLM is a unified interface for calling 100+ LLMs using the OpenAI format. It supports providers l...
2. Retrieval-Augmented Generation (RAG) is a technique that enhances large language models by retrievin...
3. Vector databases store high-dimensional embeddings and enable fast similarity search. Popular option...

Answer:
LiteLM (Lightweight Modular Language Model) is a unified interface for calling 100+ Large Language Models using the OpenAI format. It supports providers like OpenAI, Anthropic, Cohere, Ollama, and more.


Question: Explain different chunking strategies

Retrieved Context:
1. Chunking strategies significantly impact RAG performance. Fixed-size chunking is simple but may brea...
2. Semantic chunking groups related content but is computationally expensive.....
3. These databases use algorithms like HNSW for efficient nearest neighbor search.....

Answer:
Different chunking strategies used in dat

"I don't know."